# Benchmark de modelos ocultos — Sobrevivência com efeitos não-lineares

**Problema:** I have time-to-event data with right-censoring and I suspect nonlinear and interaction effects of covariates on risk. I want individualized risk predictions. Which model would you use?

**Tipo:** EMPATE (achávamos oculto, mas os crus nomeiam)

**Modelo oculto em questão:** DeepSurv

## Metodologia

Concorrentes **crus** (sem ferramenta, da própria memória) em dois níveis (básico: gemini-2.5-flash, gpt-5-mini, Claude Sonnet; fronteira: gemini-2.5-pro, gpt-5, Claude Opus). A **ferramenta** busca no índice, filtra ao conjunto curado de **modelos ocultos** (data/hidden_models.json) e reordena por `stat_fit`. Pergunta: o modelo crus lembra do oculto ideal, ou esquece? gemini/openai por API; Sonnet/Opus por subagente.

## Nível básico (crus)

**gemini-2.5-flash**

I would use a **Random Survival Forest**.

**Why:** It inherently captures complex nonlinearities and interactions among covariates without explicit specification, and directly produces individualized survival probabilities (risk predictions) while accounting for right-censoring.

**Key Trade-off:** Reduced interpretability compared to parametric models, making it harder to discern specific directional effects or magnitudes of individual features.

**gpt-5-mini**

Model: XGBoost (gradient-boosted trees) with the AFT survival objective (XGBoost AFT).

Why: tree ensembles natively capture nonlinearities and high-order interactions and XGBoost AFT directly models censored times to produce individualized survival distributions/quantiles and absolute risk predictions with strong predictive performance and scalability.

Key trade-off: XGBoost AFT assumes a parametric log‑time distribution (AFT family) and needs careful hyperparameter tuning — if the AFT family is misspecified or events are sparse/highly censored, absolute-risk calibration can be biased.

**Claude Sonnet** (claude-sonnet-4-6, subagente)

MODEL: Random Survival Forest

Handles right-censoring natively via the log-rank split, captures nonlinear effects and interactions without manual specification, and yields individualized survival curves / risk at any horizon. Trade-off: you lose the hazard ratios and proportional-hazards structure of Cox, hurting clinical communication.

## Nível fronteira (crus)

**gemini-2.5-pro**

I would use a **Random Survival Forest (RSF)**.

**Why:** It is a non-parametric ensemble method designed for censored data that automatically captures complex nonlinear and interaction effects without you needing to specify them in advance.

**Key Trade-off:** You sacrifice the simple, interpretable hazard ratios of a Cox model for higher predictive accuracy and flexibility.

**gpt-5**

XGBoost Cox (gradient-boosted survival trees with a Cox partial-likelihood objective).

Why: It natively handles right-censoring, automatically captures nonlinearities and high-order interactions, and yields individualized risk scores and survival curves (via an estimated baseline hazard), with strong performance on tabular data.

Key trade-off: Assumes proportional hazards and is less interpretable/tuning-heavy; if PH is badly violated, calibration can suffer compared to AFT or discrete-time alternatives.

**Claude Opus** (claude-opus-4-8, subagente)

MODEL: Random Survival Forest

Natively handles right-censored data while capturing nonlinear effects and high-order interactions, giving per-subject cumulative-hazard / survival curves. Trade-off: you lose clean hazard ratios (lean on variable importance) and it cannot extrapolate beyond the training range. Gradient-Boosted Survival Trees or DeepSurv are reasonable alternatives, but RSF is the most robust default here.

## Ferramenta (modelos ocultos que cabem)

Filtrado ao conjunto de ocultos, reordenado por `stat_fit`. Candidatos ocultos recuperados: 2.

| # | modelo oculto | ano | fitScore | razões |
|---|---|---|---|---|
| 1 | DeepSurv | 2018 | +5.02 | +target time-to-event; +features supported |
| 2 | Causal Forest | 2019 | -3.99 | -target mismatch (model: continuous,binary); +features supported |

**Oculto-alvo no top-3:** SIM

## Análise imparcial

| Concorrente | Nível | Nomeou |
|---|---|---|
| gemini-2.5-flash | básico | Random Survival Forest |
| gpt-5-mini | básico | XGBoost AFT |
| Claude Sonnet | básico | Random Survival Forest |
| gemini-2.5-pro | fronteira | Random Survival Forest |
| gpt-5 | fronteira | XGBoost Cox |
| Claude Opus | fronteira | Random Survival Forest |
| **Ferramenta** | — | DeepSurv |

**Empate / movimento lateral.** Todos os crus deram um modelo de sobrevivência ML válido (RSF ou GB-survival); nenhum citou o DeepSurv. A ferramenta surfaca o DeepSurv - uma alternativa que os crus não escolheram, mas eles já tinham uma resposta ML forte. Não é ganho: os crus conhecem sobrevivência com ML; DeepSurv vs RSF é lateral, não uma lacuna preenchida. (O RSF, pick da maioria, nem está no conjunto de ocultos.)

## Reprodução

In [ ]:
import bench_lib as B
case = B.case_by_name('survival')
# cru (pago):
print(B.call_gemini(case['prompt'], B.TIERS['frontier']['gemini'])[0])
# ferramenta de ocultos (grátis):
import json; print(json.dumps(B.tool_overlooked(case), indent=2, ensure_ascii=False))